# Разведочный анализ данных (EDA): Анализ настроения домовых чатов
**Проект:** VolgaFinal (Волга-IT 2026. Финал. «ИИ и анализ данных»)
**Куратор:** АИС Город (ЖКХ Ленинградской области)
**Датасеты:** `train.csv`, `dev_messages.csv`, `dev_labels.csv`, `unlabeled.csv`

### Цели и структура исследования:
1. **Аудит объемов и целостности данных:** проверка схемы, типов данных, пропусков, уникальности ID и пересечений чатов.
2. **Анализ целевых меток:** распределение `is_service`, `sentiment`, `aggression`, проверка ключевых инвариантов разметки.
3. **Анализ длины сообщений:** расчет квантилей символьной и словарной длины, контраст между жителями и объявлениями.
4. **Анализ маркеров анонимизации:** корреляция токенов (`[адрес]`, `[телефон]`, `[ссылка]` и др.) со служебными сообщениями.
5. **Частотный анализ и доменная лексика ЖКХ:** n-граммы по классам, терминология (ГВС, ХВС, парадная, УК).
6. **Анализ эмодзи по тональности:** эмоциональная окраска смайлов, использование смеющихся эмодзи в сарказме.
7. **Исследование сложных подвыборок dev:** анализ тегов `sarcasm`, `jargon`, `typos`, `context`, `emoji`, `mixed`, `obscene`.
8. **Временной анализ:** распределение активности по часам суток и дням месяца.
9. **Эталонная агрегация по чатам:** проверка `chatmood_metrics.py`, расчет трендов и пиковых дней конфликтов на dev.
10. **Архитектурные рекомендации:** выводы для схемы кросс-валидации (GroupKFold), фиче-инжиниринга и моделей.

In [1]:
import sys
import re
import warnings
from collections import Counter
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 120)

sys.path.insert(0, '..')
import chatmood_metrics as cm

print('Библиотеки успешно загружены. Модуль chatmood_metrics готов к работе.')

## 1. Загрузка данных и проверка структуры
Загружаем 4 CSV-файла: обучающий (`train.csv`), валидационный (`dev_messages.csv` + `dev_labels.csv`) и неразмеченный поток (`unlabeled.csv`). Проверяем пропуски, уникальность идентификаторов и пересечения чатов.

In [2]:
train = pd.read_csv('../data/train.csv', sep=';')
dev_msg = pd.read_csv('../data/dev_messages.csv', sep=';')
dev_lbl = pd.read_csv('../data/dev_labels.csv', sep=';')
unlabeled = pd.read_csv('../data/unlabeled.csv', sep=';')

dev = pd.merge(dev_msg, dev_lbl, on='message_id')

summary_df = pd.DataFrame([
    {'Набор': 'train.csv', 'Строк': len(train), 'Колонок': train.shape[1], 'Чатов': train['chat_id'].nunique(), 'Пользователей': train['user_id'].nunique(), 'Пропуски': train.isnull().sum().sum()},
    {'Набор': 'dev_messages.csv', 'Строк': len(dev_msg), 'Колонок': dev_msg.shape[1], 'Чатов': dev_msg['chat_id'].nunique(), 'Пользователей': dev_msg['user_id'].nunique(), 'Пропуски': dev_msg.isnull().sum().sum()},
    {'Набор': 'dev_labels.csv', 'Строк': len(dev_lbl), 'Колонок': dev_lbl.shape[1], 'Чатов': '—', 'Пользователей': '—', 'Пропуски': f"{dev_lbl.isnull().sum().sum()} (tags)"},
    {'Набор': 'unlabeled.csv', 'Строк': len(unlabeled), 'Колонок': unlabeled.shape[1], 'Чатов': unlabeled['chat_id'].nunique(), 'Пользователей': unlabeled['user_id'].nunique(), 'Пропуски': unlabeled.isnull().sum().sum()}
])

print('=== СВОДКА ПО ДАТАСЕТАМ ===')
print(summary_df.to_string(index=False))

# Проверка уникальности message_id
assert train['message_id'].is_unique and dev_msg['message_id'].is_unique and dev_lbl['message_id'].is_unique and unlabeled['message_id'].is_unique, 'Обнаружены дубликаты message_id!'

# Анализ пересечения чатов
train_chats, dev_chats, unl_chats = set(train['chat_id']), set(dev['chat_id']), set(unlabeled['chat_id'])
print('\n=== ПЕРЕСЕЧЕНИЕ ЧАТОВ МЕЖДУ ВЫБОРКАМИ ===')
print(f'Train & Dev: {len(train_chats & dev_chats)}')
print(f'Train & Unlabeled: {len(train_chats & unl_chats)}')
print(f'Dev & Unlabeled: {len(dev_chats & unl_chats)}')
print(f'Чаты в dev-сете: {sorted(list(dev_chats))}')

## 2. Анализ целевых меток и дисбаланса классов
Изучим распределение классов для трех целевых переменных:
- `is_service`: 0 — реплика жителя, 1 — официальное объявление/рассылка/реклама.
- `sentiment`: `negative`, `neutral`, `positive` (тональность сообщения).
- `aggression`: 0 — корректное высказывание, 1 — оскорбление, мат, агрессивный выпад.

Также проверяем обязательные инварианты разметки:
1. Для служебных сообщений (`is_service == 1`) строго `sentiment = neutral` и `aggression = 0`.
2. Вложенность агрессии: агрессия возможна только при негативной тональности (`sentiment == negative`).

In [3]:
def print_distributions(df, title):
    print(f'=== {title} (Всего: {len(df)}) ===')
    svc_counts = df['is_service'].value_counts(normalize=True).round(4) * 100
    print(f'is_service (%): 0 (жители) = {svc_counts.get(0, 0):.2f}%, 1 (служебные) = {svc_counts.get(1, 0):.2f}%')
    
    res_df = df[df['is_service'] == 0]
    sent_res = res_df['sentiment'].value_counts(normalize=True).round(4) * 100
    print(f'sentiment среди жителей (%): neutral = {sent_res.get("neutral", 0):.2f}%, negative = {sent_res.get("negative", 0):.2f}%, positive = {sent_res.get("positive", 0):.2f}%')
    
    agg_res = res_df['aggression'].value_counts(normalize=True).round(4) * 100
    print(f'aggression среди жителей (%): 0 = {agg_res.get(0, 0):.2f}%, 1 = {agg_res.get(1, 0):.2f}%')
    
    print('\nМатрица sentiment x aggression (только жители):')
    print(pd.crosstab(res_df['sentiment'], res_df['aggression'], margins=True))
    print()

print_distributions(train, 'TRAIN')
print_distributions(dev, 'DEV')

# Проверка жестких инвариантов
train_svc_bad = train[(train['is_service'] == 1) & ((train['sentiment'] != 'neutral') | (train['aggression'] != 0))]
dev_svc_bad = dev[(dev['is_service'] == 1) & ((dev['sentiment'] != 'neutral') | (dev['aggression'] != 0))]
print(f'Нарушений инварианта служебных сообщений в train: {len(train_svc_bad)}, в dev: {len(dev_svc_bad)}')

train_agg_bad = train[(train['sentiment'] != 'negative') & (train['aggression'] == 1)]
dev_agg_bad = dev[(dev['sentiment'] != 'negative') & (dev['aggression'] == 1)]
print(f'Нарушений инварианта агрессии (aggression=1 без negative) в train: {len(train_agg_bad)}, в dev: {len(dev_agg_bad)}')

## 3. Анализ длины сообщений
Рассчитаем распределение длины текстов по квантилям (0%, 25%, 50%, 75%, 90%, 95%, 99%, 100%) в символах и словах. Сравним длину между жителями и объявлениями, а также по тональностям.

In [4]:
for d in [train, dev, unlabeled]:
    d['char_len'] = d['text'].astype(str).str.len()
    d['word_len'] = d['text'].astype(str).str.split().apply(len)

qs = [0.0, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0]
len_df = pd.DataFrame({
    'train_chars': train['char_len'].quantile(qs).round(1),
    'train_words': train['word_len'].quantile(qs).round(1),
    'dev_chars': dev['char_len'].quantile(qs).round(1),
    'dev_words': dev['word_len'].quantile(qs).round(1),
    'unlabeled_chars': unlabeled['char_len'].quantile(qs).round(1),
    'unlabeled_words': unlabeled['word_len'].quantile(qs).round(1)
})

print('=== КВАНТИЛИ ДЛИНЫ СООБЩЕНИЙ ===')
print(len_df.to_string())

print('\n=== СРАВНЕНИЕ ДЛИНЫ: СЛУЖЕБНЫЕ vs ЖИТЕЛИ (train) ===')
print(train.groupby('is_service')[['char_len', 'word_len']].agg(['mean', 'median', 'std', 'max']).round(1))

print('\n=== СРАВНЕНИЕ ДЛИНЫ ПО ТОНАЛЬНОСТИ (только жители, train) ===')
print(train[train['is_service'] == 0].groupby('sentiment')[['char_len', 'word_len']].agg(['mean', 'median', 'std', 'max']).round(1))

## 4. Анализ маркеров анонимизации и знаков препинания
В данных персональные данные заменены специальными токенами:
`[имя]`, `[телефон]`, `[адрес]`, `[квартира]`, `[грз]`, `[email]`, `[ссылка]`, `[ник]`, `[номер]`.
Проверим их частоту и способность дифференцировать служебные сообщения от реплик жителей.

In [5]:
markers = ['[имя]', '[телефон]', '[адрес]', '[квартира]', '[грз]', '[email]', '[ссылка]', '[ник]', '[номер]']
marker_rows = []

for m in markers:
    tr_cnt = train['text'].str.contains(re.escape(m)).sum()
    dev_cnt = dev['text'].str.contains(re.escape(m)).sum()
    s_cnt = train[train['is_service'] == 1]['text'].str.contains(re.escape(m)).sum()
    r_cnt = train[train['is_service'] == 0]['text'].str.contains(re.escape(m)).sum()
    prob_svc = (s_cnt / tr_cnt * 100) if tr_cnt > 0 else 0
    marker_rows.append({
        'Маркер': m,
        'Train шт': tr_cnt,
        'Train %': round(tr_cnt / len(train) * 100, 2),
        'Dev шт': dev_cnt,
        'Dev %': round(dev_cnt / len(dev) * 100, 2),
        'В объявлениях (%)': round(s_cnt / 624 * 100, 2),
        'У жителей (%)': round(r_cnt / 3366 * 100, 2),
        'P(is_service | маркер) %': round(prob_svc, 1)
    })

print('=== СТАТИСТИКА МАРКЕРОВ АНОНИМИЗАЦИИ ===')
print(pd.DataFrame(marker_rows).to_string(index=False))

# Анализ пунктуации
def punct_stats(s):
    return pd.Series({
        'Восклицания (%)': (s.str.contains('!').mean() * 100).round(2),
        'Вопросы (%)': (s.str.contains(r'\?').mean() * 100).round(2),
        'CAPS (%)': (s.apply(lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1)).mean() * 100).round(2)
    })

print('\n=== ПУНКТУАЦИЯ ПО ТОНАЛЬНОСТИ (Жители, train) ===')
print(train[train['is_service'] == 0].groupby('sentiment')['text'].apply(punct_stats).unstack().to_string())

print('\n=== ПУНКТУАЦИЯ ПО АГРЕССИИ (Жители, train) ===')
print(train[train['is_service'] == 0].groupby('aggression')['text'].apply(punct_stats).unstack().to_string())

## 5. Частотный анализ N-грамм и доменного словаря ЖКХ
Извлекаем наиболее частые униграммы и биграммы для служебных объявлений, позитива, негатива и агрессии.

In [6]:
def get_top_terms(texts, top_k=10, ngram_range=(1, 2)):
    vec = CountVectorizer(min_df=2, ngram_range=ngram_range, token_pattern=r'(?u)[а-яёА-ЯЁa-zA-Z\[\]]{3,}')
    X = vec.fit_transform(texts)
    counts = X.sum(axis=0).A1
    vocab = vec.get_feature_names_out()
    best_idx = counts.argsort()[::-1][:top_k]
    return [f'{vocab[i]} ({counts[i]})' for i in best_idx]

print('Топ n-грамм: Служебные сообщения:')
print(', '.join(get_top_terms(train[train['is_service'] == 1]['text'], 8)))

print('\nТоп n-грамм: Позитивные реплики:')
print(', '.join(get_top_terms(train[(train['is_service'] == 0) & (train['sentiment'] == 'positive')]['text'], 8)))

print('\nТоп n-грамм: Негативные реплики:')
print(', '.join(get_top_terms(train[(train['is_service'] == 0) & (train['sentiment'] == 'negative')]['text'], 8)))

print('\nТоп n-грамм: Агрессивные выпады:')
print(', '.join(get_top_terms(train[(train['is_service'] == 0) & (train['aggression'] == 1)]['text'], 8)))

## 6. Анализ эмодзи по тональности
Выделим частоты эмодзи в текстах жителей и их связь с тональностью. Особое внимание уделим использованию смеющихся эмодзи (`🤣`, `😂`) при сарказме.

In [7]:
emoji_rx = re.compile(r'[\U00010000-\U0010ffff]', flags=re.UNICODE)

def extract_all_emojis(series):
    all_e = []
    for t in series:
        all_e.extend(emoji_rx.findall(str(t)))
    return all_e

train_emojis = extract_all_emojis(train['text'])
print(f'Всего эмодзи в train: {len(train_emojis)}, уникальных: {len(set(train_emojis))}')

for sent in ['positive', 'negative', 'neutral']:
    sub_e = extract_all_emojis(train[(train['is_service'] == 0) & (train['sentiment'] == sent)]['text'])
    top5 = Counter(sub_e).most_common(5)
    top_str = ' '.join([f'{k}:{v}' for k, v in top5])
    print(f'Эмодзи sentiment={sent:8s} (Всего: {len(sub_e):3d}): {top_str}')

## 7. Анализ сложных подвыборок dev (`tags`)
В `dev_labels.csv` присутствуют теги сложных случаев:
`sarcasm`, `jargon`, `typos`, `context`, `emoji`, `mixed`, `obscene`.
Изучим их распределение, связь с тональностью и агрессией, а также рассмотрим примеры.

In [8]:
raw_tags = []
for t in dev['tags'].dropna():
    raw_tags.extend(t.split('|'))

tag_counts = Counter(raw_tags)
tag_stat_rows = []
for tag, cnt in tag_counts.most_common():
    sub = dev[dev['tags'].fillna('').str.split('|').apply(lambda x: tag in x)]
    s = sub['sentiment'].value_counts()
    tag_stat_rows.append({
        'Тег': tag,
        'Всего': cnt,
        'Доля dev (%)': round(cnt / len(dev) * 100, 2),
        'Negative': s.get('negative', 0),
        'Neutral': s.get('neutral', 0),
        'Positive': s.get('positive', 0),
        'Aggression=1': (sub['aggression'] == 1).sum()
    })

print('=== СТАТИСТИКА ТЕГОВ DEV_LABELS ===')
print(pd.DataFrame(tag_stat_rows).to_string(index=False))

print('\n=== ПРИМЕРЫ САРКАЗМА (sarcasm) ===')
for idx, r in dev[dev['tags'].fillna('').str.contains('sarcasm')].head(2).iterrows():
    print(f'[{r["sentiment"]}, agg={r["aggression"]}] {r["text"][:100]}...')

print('\n=== ПРИМЕРЫ ОБСЦЕННОЙ ЛЕКСИКИ (obscene) ===')
for idx, r in dev[dev['tags'].fillna('').str.contains('obscene')].head(2).iterrows():
    print(f'[{r["sentiment"]}, agg={r["aggression"]}] {r["text"][:100]}...')

## 8. Временной анализ и динамика сообщений
Проверим временные интервалы датасетов, распределение активности по часам суток и активность чатов в `unlabeled.csv`.

In [9]:
for d in [train, dev, unlabeled]:
    d['dt'] = pd.to_datetime(d['timestamp'])
    d['hour'] = d['dt'].dt.hour

print('=== ВРЕМЕННЫЕ ДИАПАЗОНЫ ===')
for name, d in [('train', train), ('dev', dev), ('unlabeled', unlabeled)]:
    print(f'{name:10s}: {d["dt"].min()}  —  {d["dt"].max()} (Дней: {(d["dt"].max() - d["dt"].min()).days})')

unl_chats_stat = unlabeled['chat_id'].value_counts()
print('\n=== АКТИВНОСТЬ ЧАТОВ В UNLABELED.CSV (105 чатов) ===')
print(f'Всего сообщений: {len(unlabeled)}, Мин на чат: {unl_chats_stat.min()}, Медиана: {unl_chats_stat.median():.0f}, Среднее: {unl_chats_stat.mean():.1f}, Макс: {unl_chats_stat.max()}')

## 9. Эталонный расчёт агрегатов на dev (`chatmood_metrics.py`)
Вычисляем суточные агрегаты `chat_daily.csv` и сводку по чатам `chat_summary.csv` эталонным кодом жюри.

In [10]:
daily_gold, summary_gold = cm.gold_aggregates(dev_msg, dev_lbl)

print('=== ЭТАЛОННАЯ СВОДКА ПО ЧАТАМ DEV (chat_summary) ===')
print(summary_gold.to_string(index=False))

print('\n=== ДЕТАЛИЗАЦИЯ ДНЕВНЫХ АГРЕГАТОВ DEV (chat_daily) ===')
for c_id, g in daily_gold.groupby('chat_id'):
    print(f'Чат {c_id}: {len(g)} дней активности, {g["n_resident"].sum()} жителей, mood_index avg={g["mood_index"].mean():.2f}, neg_share avg={g["negative_share"].mean():.2f}')

## 10. Ключевые инсайты и архитектурные рекомендации

### 1. Архитектура валидации: GroupKFold по `chat_id`
- **Факт:** Пересечение чатов между `train`, `dev` и `unlabeled` равно 0 (строгая изоляция). Тестовый набор жюри будет содержать 22 совершенно новых дома.
- **Вывод:** Random Split недопустим (утечка локальных тем и авторов). Валидация модели должна строиться исключительно через **GroupKFold (по `chat_id`)**.

### 2. Иерархия меток и правила разметки
- **Инвариант 1 (Служебные):** Все служебные сообщения (`is_service == 1`) в 100% случаев имеют `sentiment = neutral` и `aggression = 0`.
- **Инвариант 2 (Агрессия):** Агрессия (`aggression == 1`) наблюдается **исключительно** при `sentiment == negative`. В нейтральных и позитивных репликах агрессия всегда равна 0.
- **Рекомендация:**
  1. Двухуровневый классификатор: сначала детектор `is_service`.
  2. Если `is_service == 1`, принудительно выставляем `sentiment = neutral`, `aggression = 0`.
  3. Пост-процессинг: если предсказанный sentiment $\neq$ `negative`, принудительно обнуляем `aggression = 0`.

### 3. Лингвистические признаки и домен ЖКХ
- **Маркеры:** токены `[адрес]`, `[телефон]`, `[ссылка]` — сильные маркеры служебных объявлений.
- **Длина:** позитивные реплики предельно кратки (медиана 28 символов), служебные — длинные (медиана 212 символов).
- **Сарказм:** смеющиеся смайлы (`🤣`, `😂`) часто сочетаются с негативом и возмущением.
- **Специфика ЖКХ:** термины `парадная` (СПб/ЛО), `ГВС`, `ХВС`, `УК` должны быть учтены в токенизаторе.

### 4. Ограничения скорости и ресурсов
- На скрытом тесте (~4500 сообщений) на GTX 1050 Ti требуется $\le 10$ сек / 1000 сообщений.
- Оптимальная модель: компактный `rubert-tiny2` (27M параметров) с multi-task head + быстрый TF-IDF + Logistic Regression baseline.